<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
       alt="Databricks Learning">
</div>

# LAB - Agentic ML with Genie Code

In this lab, you will independently build a bank personal loan prediction classifier using **Genie Code** in **Agent Mode**. Each task asks you to write a structured prompt, run it in Genie Code, review the generated code, and validate the output before moving on. The workflow deliberately extends the demo - you will apply the same Genie Code pattern to a new dataset and complete three skills the demo did not cover: business-driven feature engineering, multi-model comparison, and decision threshold optimization.

📌 **Note:** In the demo, a single Random Forest model was trained on a customer churn dataset. In this lab, you will engineer features from business reasoning, compare two models using the MLflow API, optimize the decision threshold using the precision-recall curve, and deploy the selected model to a real-time Databricks serving endpoint.

**Lab Outline:**

- **Task 1:** Load the bank loan dataset from the Feature Store and perform EDA — class distribution, null check, and correlation heatmap.

- **Task 2:** Apply business-driven feature engineering — remove a redundant feature identified during EDA and create a meaningful derived feature.

- **Task 3:** Train and compare two classification models (Logistic Regression and Random Forest) in the same MLflow experiment and select the best performer programmatically.

- **Task 4:** Evaluate the selected model and optimize the decision threshold using a precision-recall curve to maximize F1-score.

- **Task 5:** Register the selected model to Unity Catalog with a `champion` alias and a model description.

- **Task 6:** Deploy the registered model to a real-time Databricks serving endpoint and verify it with a live inference request.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
  <div>
    <strong style="color: #C62828; font-size: 1.1em;">Select Compute</strong>
    <p style="margin: 8px 0 0 0; color: #333;">Before starting, select the required compute environment listed below.</p>
    <ul style="margin: 12px 0 0 16px; color: #333;">
      <li><strong>Serverless Compute, Version 5</strong> — <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;">How to select an environment version</a></li>
    </ul>
    <p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>.</p>
  </div>
</div>
</div>

<div style="width: 100%; font-family: sans-serif;"><div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">  <img src="../Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">  <div style="font-size: 15pt; color: #0B2026; line-height: 1.7; margin-bottom: 16px;">    Starting the agentic ML lab on bank loan prediction? Ask Genie Code to orient yourself before you begin. Click on the genie icon <img src="../Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.  </div>  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0B2026;">    <span id="genie-query-lab" style="flex: 1;">I am working on a bank personal loan prediction ML lab in Databricks. I need to compare two classification models (Logistic Regression and Random Forest), optimize the decision threshold, and deploy the best model to a serving endpoint. What workflow would you recommend?</span>    <button onclick="      var text = document.getElementById('genie-query-lab').innerText;      var ta = document.createElement('textarea');      ta.value = text;      ta.style.position = 'fixed';      ta.style.opacity = '0';      document.body.appendChild(ta);      ta.select();      document.execCommand('copy');      document.body.removeChild(ta);      this.innerText = 'Copied!';      var btn = this;      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>  </div></div></div>

---
## How to Use This Lab

Each task follows the same pattern:

1. **Read the objective** — understand the goal and how it extends the demo
2. **Study the prompt** — each prompt is structured with context, task, and expected output
3. **Copy and paste into Genie Code Agent Mode** — use the copy button, paste, press Enter
4. **Review the generated code** — read it before running; refine the prompt if needed
5. **Run the cell** — execute and observe output
6. **Validate** — check the criteria before moving on

> 💡 **Prompts are starting points.** Try variations — observe how specificity changes the output. This is a key skill for agentic ML workflows.

---
### Classroom Setup

In [0]:
%run ../Includes/Classroom-Setup-3.2

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")

---
## Dataset Overview

**Task:** Predict whether a bank customer will accept a personal loan offer.
**Target:** `Personal_Loan` (1 = accepted, 0 = declined) — already encoded as binary.

**Tables available:**
- `bank_loan`: Age, Experience, Income, ZIP_Code, Family, CCAvg, Education, Mortgage, Securities_Account, CD_Account, Online, CreditCard, Personal_Loan
- `bank_loan_features`: Engineered features from the Feature Store — Loan_to_Income_Ratio, Monthly_Income

> 💡 **Important note:** You will discover in Task 2 that `Age` and `Experience` are nearly perfectly correlated (r ≈ 0.99) — a real-world data quality issue that requires a deliberate engineering decision.

## How to Access Genie Code (Agent Mode)

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 10px 0;">
  <strong style="color: #0d47a1;">Step-by-step:</strong>
  <ol style="margin: 8px 0 0 18px; color: #333; line-height: 1.8;">
    <li>Click the <strong>Genie Code</strong> icon on the right-hand panel (AI wand or chat bubble).</li>
    <li>Switch the mode drop-down to <strong>Agent</strong> (not the default "Assistant").</li>
    <li>Genie Code can now read notebook variables, insert cells, and reason across steps.</li>
    <li>For each task: click <strong>Copy Genie Prompt</strong>, paste, press Enter.</li>
    <li>Review the generated code before running it.</li>
  </ol>
</div>

### Custom Instructions for This Lab

The classroom setup cell above automatically wrote a `.assistant_instructions.md` file to this notebook's workspace folder. Genie Code reads this file when the Genie Code panel opens, so it is already grounded in the lab context before you send the first prompt.

<div style="border-left: 4px solid #2e7d32; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 10px 0;">
  <strong style="color: #1b5e20;">✅ Auto-configured — no action needed</strong>
  <p style="margin: 6px 0 0 0; color: #333;">The file was written by the setup script and will be loaded automatically each time you open Genie Code in this notebook. Re-running classroom setup rewrites it fresh (create or replace).</p>
</div>

<p style="margin: 14px 0 6px 0; font-weight: 600; color: #263238;">Contents of <code>.assistant_instructions.md</code> for this lab:</p>

<div style="background: #f8fafc; border: 1px solid #e5e7eb; border-radius: 8px; padding: 16px 20px; font-family: ui-monospace, monospace; font-size: 0.88rem; line-height: 1.6; margin: 0;">
You are assisting with a binary classification task to predict bank personal loan acceptance.<br><br>
Dataset: bank_loan (joined with bank_loan_features via ID)<br>
Join key: ID (left join from bank_loan to bank_loan_features)<br>
Target column: Personal_Loan (1 = accepted, 0 = declined) — already encoded as binary<br>
Catalog: {DA.catalog_name}<br>
Schema: {DA.schema_name}<br>
MLflow experiment: /Users/{DA.username}/loan_model_comparison<br>
Modeling library: scikit-learn (Pipeline-based preprocessing preferred)<br>
All experiments must be logged to MLflow with explicit parameter and metric logging.<br>
Register final models to Unity Catalog using mlflow.set_registry_uri("databricks-uc").<br>
Lab tasks: business-driven feature engineering, multi-model comparison (Logistic Regression vs Random Forest), decision threshold optimization using precision-recall curve, and deployment to a real-time Databricks serving endpoint.
</div>

---
## Task 1: Load Data and Perform Exploratory Data Analysis

### Objective
Load the bank loan dataset from both tables, join them, and explore the data. Pay particular attention to the class distribution and the correlation between `Age` and `Experience` — this will inform your feature engineering decisions in Task 2.

### What to accomplish:
- Load `bank_loan` + `bank_loan_features` → join on `ID` → `loan_df` / `loan_pdf`
- Check class distribution of `Personal_Loan`
- Identify missing values
- Plot a correlation heatmap — specifically note the Age/Experience correlation

> 💡 **Notice in the heatmap:** `Age` and `Experience` will show near-perfect correlation. This is a common real-world data quality issue — two features measuring the same underlying signal. You'll make a deliberate decision about this in Task 2.

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t1-load" onclick="copyT1Load()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t1-load" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: I am working in a Databricks notebook on a bank personal loan prediction task.
Two tables have been set up and are accessible via DA.catalog_name and DA.schema_name.

Task: Load and join the two tables into a single modeling DataFrame.

Tables:
- bank_loan: main table with customer demographics, financial features, and the Personal_Loan target
- bank_loan_features: Feature Store table with Loan_to_Income_Ratio and Monthly_Income
- Join key: ID (left join from bank_loan to bank_loan_features)

Requirements:
- Use spark.table() with f"{DA.catalog_name}.{DA.schema_name}.bank_loan" format
- Store the joined Spark DataFrame as loan_df
- Convert to pandas → store as loan_pdf
- Print: row count, column count, schema
- Display the first 10 rows

Expected output: loan_pdf with all columns including Personal_Loan target.
</pre>

<script>
function copyT1Load() {
  const text = document.getElementById("prompt-t1-load").innerText;
  const btn  = document.getElementById("copy-btn-t1-load");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 1: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# Confirm loan_pdf has Personal_Loan, bank_loan_features columns (Loan_to_Income_Ratio, Monthly_Income).


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 1. Load Data
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from databricks.feature_engineering import FeatureEngineeringClient

# Initialize Feature Engineering client
fe = FeatureEngineeringClient()

# Load the main loan table
loan_base_df = spark.table(f"{DA.catalog_name}.{DA.schema_name}.bank_loan")

# Load the feature store table
loan_features_df = spark.table(f"{DA.catalog_name}.{DA.schema_name}.bank_loan_features")

# Join on ID
loan_df = loan_base_df.join(loan_features_df, on="ID", how="left")

# Convert to pandas
loan_pdf = loan_df.toPandas()

print(f"Dataset: {loan_df.count()} rows, {len(loan_df.columns)} columns")
print("\nSchema:")
loan_df.printSchema()
display(loan_df.limit(10))
```

</div>
</details>

---
### Task 1 — EDA: Explore the Dataset

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t1-eda" onclick="copyT1EDA()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t1-eda" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: I have a pandas DataFrame called loan_pdf loaded in this notebook.
Task: Binary classification — predict Personal_Loan (1 = accepted, 0 = declined).

Task: Perform a comprehensive exploratory data analysis in a single cell.

Output 1 — Class distribution of Personal_Loan:
- Count and percentage for each class
- Bar chart: green for class 0 (Declined), blue for class 1 (Accepted)
- Print: is the dataset balanced or imbalanced?

Output 2 — Missing values:
- Null count and percentage per column (sorted descending)
- Only show columns with at least one null; otherwise print: "✅ No missing values."

Output 3 — Correlation heatmap:
- All numeric columns including Personal_Loan as target
- seaborn heatmap, annotated values (fmt='.2f'), coolwarm colormap
- Title: "Feature Correlation — Bank Personal Loan Dataset"
- After the heatmap, print the top 5 features by absolute correlation with Personal_Loan
- IMPORTANT: Specifically call out the correlation between 'Age' and 'Experience'

Expected: Three outputs that reveal class imbalance level, data quality, and which features matter most.
</pre>

<script>
function copyT1EDA() {
  const text = document.getElementById("prompt-t1-eda").innerText;
  const btn  = document.getElementById("copy-btn-t1-eda");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 1: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# Note the Age-Experience correlation from the heatmap — you will address it in Task 2.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 1. EDA — Class Distribution
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import pandas as pd
import matplotlib.pyplot as plt

class_counts = loan_pdf['Personal_Loan'].value_counts()
class_pct = loan_pdf['Personal_Loan'].value_counts(normalize=True).mul(100).round(1)

dist_summary = pd.DataFrame({'Count': class_counts, 'Percentage (%)': class_pct})
print("Personal_Loan Class Distribution:")
print(dist_summary.to_string())

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ['Declined (0)', 'Accepted (1)'],
    class_counts.sort_index().values,
    color=['#42A5F5', '#EF5350'], edgecolor='white', width=0.5
)
ax.bar_label(bars, labels=[f"{v:,}\n({p}%)" for v, p in zip(class_counts.sort_index().values, class_pct.sort_index().values)], padding=4, fontsize=11)
ax.set_title("Personal Loan Class Distribution", fontsize=13, pad=10)
ax.set_ylabel("Customer Count", fontsize=11)
ax.set_ylim(0, class_counts.max() * 1.2)
plt.tight_layout()
plt.show()
```

</div>
</details>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 1. EDA — Missing Values
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
null_counts = loan_pdf.isnull().sum()
null_pct = (null_counts / len(loan_pdf) * 100).round(2)

null_summary = pd.DataFrame({
    'Column': null_counts.index,
    'Null Count': null_counts.values,
    'Null Percentage (%)': null_pct.values
}).sort_values('Null Count', ascending=False)

null_filtered = null_summary[null_summary['Null Count'] > 0].reset_index(drop=True)
if null_filtered.empty:
    print("✅ No missing values found in the dataset.")
else:
    print(f"Columns with missing values ({len(null_filtered)} found):")
    print(null_filtered.to_string(index=False))
```

</div>
</details>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 1. EDA — Correlation Heatmap
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import seaborn as sns
import matplotlib.pyplot as plt

loan_corr_pdf = loan_pdf.copy()
# Personal_Loan is already numeric (0/1), just ensure int
loan_corr_pdf['Personal_Loan'] = loan_corr_pdf['Personal_Loan'].astype(int)

numeric_cols = loan_corr_pdf.select_dtypes(include=['number']).columns.tolist()
# Exclude ID from correlation (not a feature)
numeric_cols = [c for c in numeric_cols if c != 'ID']

corr_matrix = loan_corr_pdf[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Heatmap\n(Personal_Loan = target)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

loan_corrs = corr_matrix['Personal_Loan'].drop('Personal_Loan').abs().sort_values(ascending=False)
print("\nTop features by correlation with Personal_Loan:")
print(loan_corrs.to_string())
```

</div>
</details>

> ✅ **Validate before moving on:**
- `loan_pdf` is loaded with all expected columns
- Class imbalance confirmed: ~91% Declined, ~9% Accepted
- Age and Experience show very high correlation (r > 0.95) — noted for Task 2
- Top predictive features identified from heatmap (Income, CCAvg, CD_Account likely top 3)

---
## Task 2: Business-Driven Feature Engineering

### Objective
Apply deliberate, business-informed feature engineering — not just mechanical preprocessing. This task introduces two decisions that the demo's standard ColumnTransformer approach does not address:

1. **Feature redundancy:** `Age` and `Experience` are nearly perfectly correlated. Keeping both adds noise without information — you will drop `Experience`.
2. **Derived feature:** `Income` alone understates affordability for larger families. You will create `Income_per_FamilyMember = Income / Family` to capture per-capita financial capacity — a stronger predictor of loan acceptance.

### What to accomplish:
- Drop `Experience` (redundant with `Age`)
- Create a new derived feature: `Income_per_FamilyMember`
- Build a `ColumnTransformer` preprocessing pipeline
- Stratified 80/20 train/test split

> 💡 **This is real ML thinking.** Adding domain knowledge to feature engineering consistently outperforms applying generic transformations to all raw features.

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t2" onclick="copyT2()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t2" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: I have loan_pdf in memory. The task is binary classification to predict Personal_Loan.
From EDA, I found that Age and Experience are nearly perfectly correlated (~0.99) — Experience is redundant.
I also want to create a derived feature that captures per-capita financial capacity.

Task: Perform business-driven feature engineering and prepare splits for model training.

Step 1 — Feature reduction and derivation:
- Define y = loan_pdf['Personal_Loan'] (already 0/1)
- Define X = loan_pdf.drop(columns=['ID', 'Personal_Loan'])
- Drop 'Experience' from X (redundant with Age — confirmed by EDA)
- Add derived feature: X['Income_per_FamilyMember'] = X['Income'] / X['Family']
- Print the final feature list and the first 5 values of Income_per_FamilyMember

Step 2 — Preprocessing pipeline:
- Identify categorical_cols (object/category dtype) and numerical_cols (numeric dtype)
- Build ColumnTransformer 'preprocessor':
    * StandardScaler() for numerical_cols
    * OneHotEncoder(handle_unknown='ignore', sparse_output=False) for categorical_cols
    * remainder='drop'
- Print the feature counts for each type

Step 3 — Stratified split:
- 80/20 split, random_state=42, stratified on y
- X_train, X_test, y_train, y_test
- Print shapes and class balance in both splits

Expected: Feature matrix X with 'Experience' dropped and 'Income_per_FamilyMember' added, plus four clean train/test splits.
</pre>

<script>
function copyT2() {
  const text = document.getElementById("prompt-t2").innerText;
  const btn  = document.getElementById("copy-btn-t2");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 2: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# Confirm: Experience is not in X.columns. Income_per_FamilyMember is in X.columns.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 2. Business-Driven Feature Engineering
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

# Step 1 — Feature reduction + derived feature
y = loan_pdf['Personal_Loan']
X = loan_pdf.drop(columns=['ID', 'Personal_Loan'])
X = X.drop(columns=['Experience'])                             # drop redundant feature
X = X.copy()
X['Income_per_FamilyMember'] = X['Income'] / X['Family']      # derived feature

print(f"Features ({len(X.columns)}): {X.columns.tolist()}")
print(f"\nIncome_per_FamilyMember sample (first 5):")
print(X['Income_per_FamilyMember'].head().to_string())

# Step 2 — Preprocessing pipeline
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols   = X.select_dtypes(include=['number']).columns.tolist()

print(f"\nCategorical ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical   ({len(numerical_cols)}): {numerical_cols}")

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
], remainder='drop')

# Step 3 — Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")
print(f"Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Test  class balance: {y_test.value_counts(normalize=True).round(3).to_dict()}")
```

</div>
</details>

> ✅ **Validate before moving on:**
- `Experience` is NOT in `X.columns`
- `Income_per_FamilyMember` IS in `X.columns` with non-null values
- `preprocessor` is a `ColumnTransformer` (not yet fitted)
- Class balance is ~91%/9% in both splits — consistent with original dataset

---
## Task 3: Multi-Model Comparison

### Objective
Train two classification models and compare them using MLflow. In the demo, a single Random Forest was trained. Here you will train both a **Logistic Regression baseline** and a **Random Forest**, log them to the same MLflow experiment, and use `mlflow.search_runs()` to build a data-driven comparison.

This mirrors real ML workflows: you always establish a baseline before claiming a more complex model is "better."

### What to accomplish:
- Train Logistic Regression (simple, interpretable, interpretable baseline)
- Train Random Forest (non-linear, typically stronger on tabular data)
- Log both to the same MLflow experiment with identical metrics
- Use MLflow API to compare results and identify the better model

> 💡 **MLflow as a comparison tool.** The `mlflow.search_runs()` function returns all runs in an experiment as a pandas DataFrame — enabling programmatic comparison. This is a Databricks-native pattern for multi-model experiments.

---
### Task 3a — Train Logistic Regression (Baseline)

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t3-lr" onclick="copyT3LR()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t3-lr" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: X_train, X_test, y_train, y_test, and preprocessor are defined.
Binary classification, class-imbalanced (~91% negative, ~9% positive).

Task: Train a Logistic Regression classifier as a baseline model, logged to MLflow.

Requirements:
- Build a Pipeline: ('preprocessor', preprocessor) + ('classifier', LogisticRegression)
- Use class_weight='balanced', max_iter=1000, solver='lbfgs', random_state=42
- MLflow:
    * Set experiment: mlflow.set_experiment(f"/Users/{DA.username}/loan_model_comparison")
    * Run name: 'lr_baseline'
    * Enable autologging: mlflow.sklearn.autolog(silent=True)
    * Explicitly log: C=1.0, max_iter=1000, solver='lbfgs', class_weight='balanced' as parameters
    * Compute and log: train_accuracy, test_accuracy, and test_roc_auc as metrics
    * Log pipeline artifact as 'loan_lr_model'
    * Store run.info.run_id as lr_run_id
- After training, print: lr_run_id, test_accuracy, test_roc_auc

Expected: lr_pipeline trained, lr_run_id stored, first MLflow run in 'loan_model_comparison' experiment.
</pre>

<script>
function copyT3LR() {
  const text = document.getElementById("prompt-t3-lr").innerText;
  const btn  = document.getElementById("copy-btn-t3-lr");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 3: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# After running: navigate to Experiments → loan_model_comparison. Confirm the lr_baseline run appears.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 3a. Train Logistic Regression (Baseline)
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

mlflow.set_experiment(f"/Users/{DA.username}/loan_model_comparison")
mlflow.sklearn.autolog(log_input_examples=False, log_model_signatures=True, silent=True)

with mlflow.start_run(run_name="lr_baseline") as run:
    lr_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier',   LogisticRegression(
            C=1.0, max_iter=1000, solver='lbfgs',
            class_weight='balanced', random_state=42
        ))
    ])
    lr_pipeline.fit(X_train, y_train)

    train_acc = lr_pipeline.score(X_train, y_train)
    test_acc  = lr_pipeline.score(X_test,  y_test)
    test_auc  = roc_auc_score(y_test, lr_pipeline.predict_proba(X_test)[:, 1])

    mlflow.log_params({"C": 1.0, "max_iter": 1000, "solver": "lbfgs", "class_weight": "balanced"})
    mlflow.log_metrics({"train_accuracy": train_acc, "test_accuracy": test_acc, "test_roc_auc": test_auc})
    mlflow.sklearn.log_model(lr_pipeline, artifact_path="loan_lr_model")
    lr_run_id = run.info.run_id

print(f"LR Run ID:        {lr_run_id}")
print(f"LR Test Accuracy: {test_acc:.4f}")
print(f"LR ROC-AUC:       {test_auc:.4f}")
```

</div>
</details>

---
### Task 3b — Train Random Forest

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t3-rf" onclick="copyT3RF()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t3-rf" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: X_train, X_test, y_train, y_test, and preprocessor are defined.
lr_run_id is stored. The MLflow experiment 'loan_model_comparison' already has the LR baseline run.

Task: Train a Random Forest classifier in the same experiment for comparison.

Requirements:
- Build a Pipeline: ('preprocessor', preprocessor) + ('classifier', RandomForestClassifier)
- Use n_estimators=100, max_depth=10, class_weight='balanced', random_state=42
- MLflow:
    * Same experiment: /Users/{DA.username}/loan_model_comparison
    * Run name: 'rf_model'
    * Enable autologging: mlflow.sklearn.autolog(silent=True)
    * Explicitly log: n_estimators=100, max_depth=10, class_weight='balanced' as parameters
    * Compute and log: train_accuracy, test_accuracy, and test_roc_auc as metrics
    * Log pipeline artifact as 'loan_rf_model'
    * Store run.info.run_id as rf_run_id
- After training, print: rf_run_id, test_accuracy, test_roc_auc

Expected: rf_pipeline trained, rf_run_id stored, second run added to 'loan_model_comparison' experiment.
</pre>

<script>
function copyT3RF() {
  const text = document.getElementById("prompt-t3-rf").innerText;
  const btn  = document.getElementById("copy-btn-t3-rf");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 3: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# After running: confirm both lr_baseline and rf_model runs appear in Experiments → loan_model_comparison.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 3b. Train Random Forest
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="rf_model") as run:
    rf_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier',   RandomForestClassifier(
            n_estimators=100, max_depth=10,
            class_weight='balanced', random_state=42
        ))
    ])
    rf_pipeline.fit(X_train, y_train)

    train_acc = rf_pipeline.score(X_train, y_train)
    test_acc  = rf_pipeline.score(X_test,  y_test)
    test_auc  = roc_auc_score(y_test, rf_pipeline.predict_proba(X_test)[:, 1])

    mlflow.log_params({"n_estimators": 100, "max_depth": 10, "class_weight": "balanced"})
    mlflow.log_metrics({"train_accuracy": train_acc, "test_accuracy": test_acc, "test_roc_auc": test_auc})
    mlflow.sklearn.log_model(rf_pipeline, artifact_path="loan_rf_model")
    rf_run_id = run.info.run_id

print(f"RF Run ID:        {rf_run_id}")
print(f"RF Test Accuracy: {test_acc:.4f}")
print(f"RF ROC-AUC:       {test_auc:.4f}")
```

</div>
</details>

---
### Task 3c — Compare Models via MLflow API

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t3-compare" onclick="copyT3Compare()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t3-compare" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: Both lr_run_id and rf_run_id are stored. Both used experiment '/Users/{DA.username}/loan_model_comparison'.

Task: Compare the two trained models using the MLflow Runs API.

Requirements:
- Retrieve runs: mlflow.search_runs(experiment_names=[f"/Users/{DA.username}/loan_model_comparison"])
- Filter to only our two runs (run names: 'lr_baseline', 'rf_model')
- Build a clean comparison table with columns: Model, Test Accuracy, ROC-AUC, Run ID
- Create a side-by-side bar chart comparing test_roc_auc for both models (use matplotlib)
- Print: which model has higher ROC-AUC?
- Store the name of the better model as best_model_name (either 'lr_baseline' or 'rf_model')
- Store the corresponding pipeline as best_pipeline (either lr_pipeline or rf_pipeline)

Expected: A comparison table, a bar chart, and best_pipeline set to the winning model.
</pre>

<script>
function copyT3Compare() {
  const text = document.getElementById("prompt-t3-compare").innerText;
  const btn  = document.getElementById("copy-btn-t3-compare");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 3: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# Confirm best_pipeline is set. Note which model won on ROC-AUC and why you think that is.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 3c. Compare Models via MLflow API
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import matplotlib.pyplot as plt

# Retrieve both runs from the experiment
runs_df = mlflow.search_runs(
    experiment_names=[f"/Users/{DA.username}/loan_model_comparison"]
)

# Filter to our two runs
our_runs = runs_df[runs_df['tags.mlflow.runName'].isin(['lr_baseline', 'rf_model'])].copy()
our_runs = our_runs[['tags.mlflow.runName', 'metrics.test_accuracy', 'metrics.test_roc_auc', 'run_id']]
our_runs.columns = ['Model', 'Test Accuracy', 'ROC-AUC', 'Run ID']
our_runs = our_runs.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print("Model Comparison:")
print(our_runs[['Model', 'Test Accuracy', 'ROC-AUC']].to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2574B5', '#1B5162']
bars = ax.bar(our_runs['Model'], our_runs['ROC-AUC'], color=colors, width=0.4)
ax.bar_label(bars, labels=[f"{v:.4f}" for v in our_runs['ROC-AUC']], padding=3, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_title("Model Comparison — ROC-AUC on Test Set", fontsize=13, pad=10)
ax.set_ylabel("ROC-AUC")
plt.tight_layout()
plt.show()

# Select best model
best_model_name = our_runs.iloc[0]['Model']
best_pipeline = rf_pipeline if best_model_name == 'rf_model' else lr_pipeline
best_run_id   = rf_run_id   if best_model_name == 'rf_model' else lr_run_id

print(f"\n✅ Best model: {best_model_name} (ROC-AUC: {our_runs.iloc[0]['ROC-AUC']:.4f})")
print(f"   best_pipeline and best_run_id are set for Tasks 4 and 5.")
```

</div>
</details>

> ✅ **Validate before moving on:**
- Both `lr_run_id` and `rf_run_id` are defined
- Comparison table shows both models' Test Accuracy and ROC-AUC
- `best_pipeline` is set to the winning model
- In **MLflow Experiments UI**: both runs appear in `loan_model_comparison`

---
## Task 4: Model Selection and Decision Threshold Optimization

### Objective
Evaluate the best model in depth — then go beyond the demo's default-threshold evaluation. In production loan models, **the decision threshold (default: 0.5) is a business parameter, not a fixed constant**. Lowering it increases recall (you catch more loan opportunities) but reduces precision (more false approvals). The right threshold depends on business cost assumptions.

You will:
1. Generate standard evaluation metrics for `best_pipeline`
2. Plot a precision-recall curve across all thresholds
3. Identify the threshold that maximizes F1-score for class 1 (loan acceptors)
4. Compare default threshold vs. optimized threshold results

> 💡 **Why this matters:** The demo evaluated at threshold 0.5 and reported recall. Here, you will show that with a better threshold choice, you can materially improve recall — potentially capturing significantly more loan candidates — without unacceptable precision loss.

---
### Task 4a — Standard Evaluation at Default Threshold

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t4-eval" onclick="copyT4Eval()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t4-eval" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: best_pipeline has been selected and trained. X_test and y_test are available.
Task: Bank personal loan prediction (1 = accepted, 0 = declined).

Task: Generate standard evaluation metrics for best_pipeline at the default threshold (0.5).

Requirements:
1. Predictions: y_pred = best_pipeline.predict(X_test)
   Probabilities: y_prob = best_pipeline.predict_proba(X_test)[:, 1]

2. Plot confusion matrix (ConfusionMatrixDisplay):
   - Labels: ['Declined (0)', 'Accepted (1)']
   - Blues colormap
   - Title: "Confusion Matrix — Bank Loan (Default Threshold = 0.5)"

3. Print classification report with target_names=['Declined (0)', 'Accepted (1)']

4. Print ROC-AUC score

After the outputs, print a one-line summary:
"At threshold 0.5: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f} for class 1 (Accepted)"

Expected: Standard evaluation outputs that establish the baseline for threshold comparison.
</pre>

<script>
function copyT4Eval() {
  const text = document.getElementById("prompt-t4-eval").innerText;
  const btn  = document.getElementById("copy-btn-t4-eval");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 4: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# Note the recall for class 1 (Accepted) at threshold 0.5 — you will compare this after threshold optimization.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 4a. Standard Evaluation at Default Threshold
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score
)
import matplotlib.pyplot as plt

y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred),
    display_labels=['Declined (0)', 'Accepted (1)']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title("Confusion Matrix — Bank Loan (Default Threshold = 0.5)", fontsize=11, pad=10)
plt.tight_layout()
plt.show()

# Classification report
print("Classification Report (threshold = 0.5):")
print(classification_report(y_test, y_pred, target_names=['Declined (0)', 'Accepted (1)']))

# ROC-AUC
auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {auc:.4f}")

# One-line summary for class 1
p05 = precision_score(y_test, y_pred)
r05 = recall_score(y_test, y_pred)
f05 = f1_score(y_test, y_pred)
print(f"\nAt threshold 0.5: Precision={p05:.3f}, Recall={r05:.3f}, F1={f05:.3f} for class 1 (Accepted)")
```

</div>
</details>

---
### Task 4b — Precision-Recall Curve and Threshold Optimization

Now find the threshold that maximizes F1-score for class 1 — and compare it against the default.

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t4-thresh" onclick="copyT4Thresh()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t4-thresh" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: y_prob (predicted probabilities from best_pipeline.predict_proba(X_test)[:, 1]) is already computed.
y_test is the true labels. Business context: for a bank loan model, we want to maximize recall for
class 1 (Accepted) to capture more loan opportunities, while keeping precision reasonable.

Task: Perform decision threshold optimization for the loan prediction model.

Step 1 — Precision-Recall curve:
- Use sklearn.metrics.precision_recall_curve(y_test, y_prob) to get precision, recall, thresholds
- Plot: x=recall, y=precision
- Annotate the point at threshold=0.5 with a marker
- Annotate the point at the optimal F1 threshold with a different marker
- Add a legend and title: "Precision-Recall Curve — Bank Loan Prediction"

Step 2 — Find optimal threshold (maximizes F1 for class 1):
- For each threshold in numpy.linspace(0.1, 0.9, 81):
    * Apply threshold: y_pred_t = (y_prob >= threshold).astype(int)
    * Compute F1 for class 1: f1_score(y_test, y_pred_t, pos_label=1)
- Find the threshold with the highest F1
- Store as optimal_threshold

Step 3 — Apply optimal threshold and compare:
- Generate y_pred_opt = (y_prob >= optimal_threshold).astype(int)
- Print classification report at optimal threshold
- Print a comparison table:
    | Metric      | Default (0.5) | Optimal ({optimal_threshold:.2f}) |
    |-------------|---------------|-----------------------------------|
    | Precision   | ...           | ...                               |
    | Recall      | ...           | ...                               |
    | F1-Score    | ...           | ...                               |

Expected: A precision-recall curve, the optimal threshold value, and a clear before/after comparison.
</pre>

<script>
function copyT4Thresh() {
  const text = document.getElementById("prompt-t4-thresh").innerText;
  const btn  = document.getElementById("copy-btn-t4-thresh");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 4: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# Compare recall at default vs optimal threshold. How many more loan opportunities does the optimized threshold capture?


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 4b. Precision-Recall Curve and Threshold Optimization
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import numpy as np
from sklearn.metrics import (
    precision_recall_curve, precision_score, recall_score, f1_score, classification_report
)
import matplotlib.pyplot as plt

# Step 1 — Precision-Recall curve
precision_pts, recall_pts, thresh_pts = precision_recall_curve(y_test, y_prob)

# Find point at default threshold 0.5
idx_05 = np.argmin(np.abs(thresh_pts - 0.5))

# Step 2 — Find optimal threshold
thresholds = np.linspace(0.1, 0.9, 81)
f1_scores  = [f1_score(y_test, (y_prob >= t).astype(int), pos_label=1) for t in thresholds]
optimal_threshold = thresholds[np.argmax(f1_scores)]

# Find the matching point on PR curve
idx_opt = np.argmin(np.abs(thresh_pts - optimal_threshold))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall_pts, precision_pts, lw=2, color='#2574B5', label='PR Curve')
ax.scatter(recall_pts[idx_05],  precision_pts[idx_05],
           s=120, zorder=5, color='orange', marker='o',
           label=f'Threshold = 0.50 (F1={f1_score(y_test,(y_prob>=0.5).astype(int)):.3f})')
ax.scatter(recall_pts[idx_opt], precision_pts[idx_opt],
           s=120, zorder=5, color='green', marker='*',
           label=f'Optimal threshold = {optimal_threshold:.2f} (F1={max(f1_scores):.3f})')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curve — Bank Loan Prediction', fontsize=12, pad=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Step 3 — Compare thresholds
y_pred_opt = (y_prob >= optimal_threshold).astype(int)
print(f"\nClassification Report at Optimal Threshold ({optimal_threshold:.2f}):")
print(classification_report(y_test, y_pred_opt, target_names=['Declined (0)', 'Accepted (1)']))

# Comparison table
p_def, r_def, f_def = precision_score(y_test,y_pred), recall_score(y_test,y_pred), f1_score(y_test,y_pred)
p_opt, r_opt, f_opt = precision_score(y_test,y_pred_opt), recall_score(y_test,y_pred_opt), f1_score(y_test,y_pred_opt)

print(f"{'Metric':<15} {'Default (0.50)':>15} {'Optimal ({:.2f})'.format(optimal_threshold):>18}")
print("-"*50)
for name, d, o in [("Precision", p_def, p_opt), ("Recall", r_def, r_opt), ("F1-Score", f_def, f_opt)]:
    print(f"{name:<15} {d:>15.3f} {o:>18.3f}")
print(f"\n✅ Optimal threshold: {optimal_threshold:.2f}")
```

</div>
</details>

> ✅ **Validate before moving on:**
- Precision-recall curve is plotted with both threshold points annotated
- `optimal_threshold` is defined and differs from 0.5
- Comparison table shows the recall improvement at the optimized threshold
- You can explain the business trade-off: higher recall means more loan opportunities captured

---
## Task 5: Register the Selected Model to Unity Catalog

### Objective
Register `best_pipeline` (the model selected in Task 3) to Unity Catalog. This is the model that will be served in Task 6.

### What to accomplish:
- Register from `best_run_id` to Unity Catalog
- Set alias: `champion`
- Include a description that references the model selection process
- Confirm registration in Unity Catalog → Models

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t5" onclick="copyT5()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t5" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: best_run_id is defined. The model artifact path is runs:/{best_run_id}/loan_rf_model
(or loan_lr_model if Logistic Regression was selected — use the correct artifact name).
DA.catalog_name and DA.schema_name are defined.

Task: Register the selected model to Unity Catalog with a production alias.

Requirements:
- mlflow.set_registry_uri("databricks-uc")
- Register:
    * model_uri = f"runs:/{best_run_id}/loan_rf_model"
    * name = f"{DA.catalog_name}.{DA.schema_name}.loan_rf_classifier"
- Use MlflowClient to:
    * Set alias "champion" on the registered version
    * Add description:
      "Best-performing model for bank personal loan prediction.
       Selected via multi-model comparison (Random Forest vs Logistic Regression baseline).
       Feature engineering: dropped redundant Experience, added Income_per_FamilyMember.
       Evaluated with precision-recall threshold optimization."
- Print: model name, version, alias

Expected: Model appears in Unity Catalog with version 1 and 'champion' alias.
</pre>

<script>
function copyT5() {
  const text = document.getElementById("prompt-t5").innerText;
  const btn  = document.getElementById("copy-btn-t5");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 5: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# Navigate to Unity Catalog → Models → confirm name, version 1, and 'champion' alias.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 5. Register the Model
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

model_name = f"{DA.catalog_name}.{DA.schema_name}.loan_rf_classifier"
model_uri  = f"runs:/{run_id}/loan_rf_model"

print(f"Registering model: {model_name}")

# Register
registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)
model_version = registered_model.version
print(f"✅ Registered version: {model_version}")

# Set champion alias
client.set_registered_model_alias(name=model_name, alias="champion", version=model_version)
print(f"✅ Alias set: champion → version {model_version}")

# Add description
client.update_model_version(
    name=model_name,
    version=model_version,
    description=(
        "Random Forest classifier for bank personal loan prediction. "
        "Trained via Genie Code (Agent Mode) in the ML Model Development course lab. "
        "Features: customer demographics, financial profile (Income, CCAvg, Mortgage), service usage."
    )
)
print(f"✅ Description added")
print(f"\nNavigate to: Unity Catalog → Models → {model_name}")
```

</div>
</details>

> ✅ **Validate before moving on:**
- Model appears in **Unity Catalog → Models** under your catalog and schema
- Version 1 has the `champion` alias
- Description references the model selection and feature engineering steps

---
## Task 6: Deploy and Test the Serving Endpoint

### Objective
Deploy the registered model to a real-time Databricks serving endpoint, verify it reaches `READY` state, and **send a sample inference request** to confirm the endpoint returns predictions. This closes the full ML lifecycle loop.

### What to accomplish:
- Create a real-time serving endpoint using `WorkspaceClient`
- Wait for `READY` state
- Send a sample inference request using `X_test` records
- Confirm the endpoint returns loan predictions (0 or 1)

> ⚠️ **Endpoint creation takes 3–8 minutes.** The deployment cell blocks until `READY`. After the endpoint is live, the inference test cell confirms it responds correctly.

---
### Task 6a — Create the Serving Endpoint

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t6-deploy" onclick="copyT6Deploy()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t6-deploy" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: A model is registered in Unity Catalog as:
{DA.catalog_name}.{DA.schema_name}.loan_rf_classifier  (alias: champion, version 1)
DA.catalog_name, DA.schema_name are defined. X_test is available.

Task: Deploy the registered model to a Databricks real-time serving endpoint.

Requirements:
1. from databricks.sdk import WorkspaceClient
   from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
2. w = WorkspaceClient()
3. endpoint_name = f"loan-rf-{DA.schema_name}".replace("_","-").replace(".","-")
   (lowercase with hyphens — no underscores or dots)
4. Call w.serving_endpoints.create_and_wait():
   - name: endpoint_name
   - config: EndpointCoreConfigInput with ServedEntityInput:
       * entity_name: full UC model name
       * entity_version: "1"
       * scale_to_zero_enabled: True
       * workload_size: "Small"
5. Print endpoint name and state after READY

Expected: Endpoint in Serving UI with READY status.
</pre>

<script>
function copyT6Deploy() {
  const text = document.getElementById("prompt-t6-deploy").innerText;
  const btn  = document.getElementById("copy-btn-t6-deploy");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 6: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# After the cell completes: navigate to Serving → Endpoints → confirm READY status.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 6a. Create the Serving Endpoint
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()

model_name    = f"{DA.catalog_name}.{DA.schema_name}.loan_rf_classifier"
endpoint_name = f"loan-rf-{DA.schema_name}".replace("_", "-").replace(".", "-")

print(f"Creating endpoint: {endpoint_name}")
print(f"Model            : {model_name}@champion")

endpoint = w.serving_endpoints.create_and_wait(
    name=endpoint_name,
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=model_name,
                entity_version="1",
                scale_to_zero_enabled=True,
                workload_size="Small",
            )
        ]
    ),
)

print(f"\n✅ Endpoint '{endpoint.name}' is {endpoint.state.ready.value}")
print(f"   Navigate to: Serving → Endpoints → {endpoint_name}")
```

</div>
</details>

---
### Task 6b — Test the Endpoint with a Sample Inference Request

Now that the endpoint is live, verify it works by sending real records from your test set.

<div style="border-left:3px solid #42a5f5; background:#e3f2fd; padding:10px 14px; border-radius:4px; margin:0 0 12px 0; font-size:0.88rem;">
<strong style="color:#0d47a1;">How to run this prompt:</strong>
<ol style="margin:4px 0 0 16px; color:#333; line-height:1.6;">
<li>Open <strong>Genie Code</strong> panel → switch mode to <strong>Agent</strong></li>
<li>Click <strong>Copy Genie Prompt</strong> below and paste into the chat</li>
<li>Press Enter — Genie Code generates and inserts a cell</li>
<li>Review the code, then run it</li>
</ol>
</div>

<button id="copy-btn-t6-infer" onclick="copyT6Infer()" style="background:#ff6b3d; color:white; border:none; padding:8px 14px; border-radius:6px; cursor:pointer; font-weight:600; margin-bottom:10px;">
Copy Genie Prompt
</button>

<pre id="prompt-t6-infer" style="font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.5; white-space:pre-wrap;">
Context: A Databricks serving endpoint named endpoint_name is READY.
X_test is available as a pandas DataFrame with the same features the model was trained on.
w = WorkspaceClient() is already initialized.

Task: Send a sample inference request to the deployed endpoint and display the predictions.

Requirements:
1. Take the first 5 rows of X_test
2. Convert to a list of dicts: sample_records = X_test.head(5).to_dict(orient='records')
3. Call: response = w.serving_endpoints.query(name=endpoint_name, dataframe_records=sample_records)
4. Extract predictions from response.predictions
5. Print a table showing:
   - Row index
   - Key features (Income, Income_per_FamilyMember, CCAvg)
   - Predicted label (0 = Declined, 1 = Accepted)
   - True label from y_test

Expected: A table showing 5 sample records, their predicted loan decisions, and the true labels.
</pre>

<script>
function copyT6Infer() {
  const text = document.getElementById("prompt-t6-infer").innerText;
  const btn  = document.getElementById("copy-btn-t6-infer");
  navigator.clipboard.writeText(text).then(() => {
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  }).catch(() => {
    const ta = document.createElement('textarea'); ta.value = text;
    ta.style.position = 'fixed'; ta.style.left = '-9999px';
    document.body.appendChild(ta); ta.focus(); ta.select();
    document.execCommand('copy'); document.body.removeChild(ta);
    btn.innerText = '\u2705 Copied!'; setTimeout(() => { btn.innerText = 'Copy Genie Prompt'; }, 2000);
  });
}
</script>

In [0]:
# TODO — Task 6: Use Genie Code (Agent Mode) to generate and run the cell above.
# After running, validate the output against the criteria below.
# Confirm the endpoint returns 0/1 predictions. Check if predictions match y_test for the sample rows.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">SOLUTION</span> Task 6b. Test the Endpoint with Sample Inference Request
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import pandas as pd

# Take 5 test samples for inference
sample_x  = X_test.head(5).reset_index(drop=True)
sample_y  = y_test.head(5).reset_index(drop=True)

# Convert to list of records for the serving endpoint
sample_records = sample_x.to_dict(orient='records')

# Query the endpoint
response = w.serving_endpoints.query(
    name=endpoint_name,
    dataframe_records=sample_records
)

# Extract predictions
predictions = response.predictions

# Display comparison table
result_df = pd.DataFrame({
    'Income': sample_x['Income'].values,
    'Income_per_FM': sample_x['Income_per_FamilyMember'].round(2).values,
    'CCAvg': sample_x['CCAvg'].values,
    'Predicted': predictions,
    'Actual': sample_y.values,
    'Correct': ['✅' if p == a else '❌' for p, a in zip(predictions, sample_y.values)]
})

print(f"Endpoint: {endpoint_name}")
print(f"\nSample Inference Results (5 rows):")
print(result_df.to_string(index=False))
print(f"\n✅ Endpoint is live and returning predictions.")
```

</div>
</details>

> ✅ **Validate before moving on:**
- Endpoint returns a response without error
- Predictions are 0 or 1 (loan declined or accepted)
- The Correct column shows at least some ✅ matches — confirming the model is working
- Navigate to **Serving → Endpoints → {endpoint_name}** to view the request log

---
## Conclusion

You have completed an end-to-end agentic ML workflow — and gone **further than the demo** with three new skills.

| Task | What You Did | New skill vs. Demo |
|------|-------------|-------------------|
| Task 1 — Load & EDA | Loaded bank loan data, explored class balance, nulls, correlations | EDA on new dataset |
| Task 2 — Feature Engineering | Dropped redundant `Experience`, created `Income_per_FamilyMember` derived feature | **Business-driven feature decisions** |
| Task 3 — Multi-Model Comparison | Trained Logistic Regression + Random Forest, compared via `mlflow.search_runs()` | **Multi-model comparison + MLflow API** |
| Task 4 — Threshold Optimization | Generated precision-recall curve, found optimal decision threshold | **Threshold optimization for business context** |
| Task 5 — Registration | Registered the selected model to Unity Catalog with `champion` alias | UC model lifecycle |
| Task 6 — Deployment + Inference | Deployed to serving endpoint, tested with live inference request | **End-to-end serving verification** |

**Key takeaways:**
- Feature engineering with business logic (dropping correlated features, creating derived features) is more powerful than generic preprocessing
- Comparing models via `mlflow.search_runs()` is a Databricks-native pattern for multi-model experiments
- The decision threshold is a **business parameter** — not a fixed constant. Optimizing it can materially improve business outcomes
- Closing the full loop (deploy + test inference) confirms the entire ML system works end-to-end

> 💡 **Reflect:** Where in this lab did Genie Code produce the most useful output? Where did your prompt specificity make the biggest difference?

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>